# Lecture 2.6 — Cloning Agents with `.clone()` for Persona Variants

**Course:** OpenAI Agents SDK — Complete Course  
**Section:** 02 — Agents: Configuration & Behaviour  

In this notebook you will learn how to use the `Agent.clone()` method to create multiple agent variants from a single base definition. This is the clean, maintainable alternative to copy-pasting `Agent(...)` blocks and manually keeping them in sync.

## Cell 1 — Install the SDK

📌 **Notebook update notice:** this lecture's video and markdown reference `openai-agents==0.17.4` as the tested version. Since recording, a downstream dependency change (`openai>=2.45.0`, released July 9, 2026) broke `openai-agents` versions below 0.18.1 — `Runner.run()` will fail on the version stated in the video. This notebook has been updated to pin `openai-agents==0.18.3`, which fixes the issue without changing any of the code or concepts taught in the lecture. Please use the version pinned below, not the one mentioned in the recording.

We pin the SDK to a specific version for reproducibility. Every cell in this notebook was tested against `openai-agents==0.18.3`. If you want to use the latest published version instead, run `pip install openai-agents` without a version pin, or substitute your preferred version below.

**Note:** This install cell is scoped to the current notebook session only. It does not affect other notebooks or your system environment.

In [1]:
# Pinned for reproducibility. Updated after recording — see the
# notice above. Originally pinned to 0.17.4 as stated in the
# video; updated to 0.18.3 to fix a breaking change introduced
# by openai>=2.45.0 (July 9, 2026).
# To use the latest version instead, run: pip install openai-agents
!pip install openai-agents==0.18.3 -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 880.8/880.8 kB 13.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.6/142.6 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 223.4/223.4 kB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 5.7 MB/s eta 0:00:00


## Cell 2 — API Key Setup (Google Colab Secrets)

We load the OpenAI API key from **Google Colab Secrets** — a secure, project-level store that keeps credentials out of your notebook code.

**Step-by-step Colab instructions:**
1. Click the 🔑 **Secrets** icon in the left sidebar (or go to **Tools → Secrets**).
2. Click **+ Add new secret**.
3. Set **Name** to `OPENAI_API_KEY`.
4. Paste your OpenAI API key as the **Value**.
5. Toggle **Notebook access** to **ON** for this notebook.
6. Run the cell below — it will read the secret and write it to the environment variable.

**Local users:** Instead of running this cell, set the environment variable in your terminal before launching Jupyter:
```bash
export OPENAI_API_KEY="sk-..."
```
Then comment out or skip the cell below.

In [2]:
from google.colab import userdata
import os

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

## Cell 3 — `MODEL_NAME` Variable

We declare a single `MODEL_NAME` variable here and use it in every `Agent` definition throughout the notebook. This means:

- You change the model in **one place** and every agent picks it up automatically.
- No hardcoded model strings buried inside individual agent definitions.
- It mirrors how production code should work — configuration at the top, not scattered through the logic.

See the full list of available OpenAI models at: https://platform.openai.com/docs/models

In [3]:
# See latest models at: https://platform.openai.com/docs/models
MODEL_NAME = "gpt-5.4-mini"

## Cell 4 — Imports

Here is what we import and why:

| Import | Purpose |
|---|---|
| `Reasoning` | Controls the reasoning effort level inside `ModelSettings`. Imported from `openai.types.shared`. |
| `Agent` | The core class we are cloning in this lecture. |
| `ModelSettings` | Bundles model-level configuration (temperature, reasoning effort, verbosity, etc.) into a single object passed to `Agent`. |
| `Runner` | Executes agents. We always use `await Runner.run(...)` in notebooks — `run_sync()` raises a `RuntimeError` because Jupyter/Colab already have an event loop running. |
| `function_tool` | Decorator that wraps a Python function as a tool an agent can call. **This is imported here only to demonstrate the shallow copy behaviour of `clone()` in Cell 10.** The full `function_tool` teaching happens in Section 3. |

In [5]:
from openai.types.shared import Reasoning
from agents import Agent, ModelSettings, Runner, function_tool

## What is `clone()`?

### The problem `clone()` solves

Suppose you want three writing assistants — a default one, a pirate-themed one, and a formal academic one. Without `clone()`, you would copy and paste three nearly identical `Agent(...)` blocks and manually keep them in sync whenever you change the model or model settings. That is fragile and repetitive.

`Agent.clone(**kwargs)` solves this cleanly: define the base agent **once**, then derive variants from it by overriding only the fields that differ.

### How `clone()` works internally

`clone()` uses Python's `dataclasses.replace()` under the hood, which performs a **shallow copy**. That means:

- **Non-overridden fields** are inherited from the original exactly as-is — same object references, not copies.
- **Overridden fields** replace the original value in the new agent.
- **Mutable attributes** (`tools`, `handoffs`): a new list object is only created for a field if you explicitly pass it in `kwargs`. If you do not, the clone and the original share the same list object. The *contents* of those lists (tool functions, handoff objects) are always shared references — this is the nature of a shallow copy.

**Practical rule:** If you plan to modify `tools` or `handoffs` on a clone independently of the original, always pass a new list when calling `clone()`. The unpacking pattern `tools=[*base.tools, new_tool]` is the clean way to extend a list while cloning.

### Model change auto-update behaviour

When you clone an agent and pass a new `model` without also passing `model_settings`, the SDK checks whether the current `model_settings` are the implicit defaults for the *current* model. If they are, it automatically updates `model_settings` to the implicit defaults for the *new* model. This prevents stale model-specific settings from carrying forward when you upgrade models.

If you pass **both** `model` **and** `model_settings` explicitly, the auto-update does **not** trigger — your explicit `model_settings` are used as-is.

### Summary

| What you do | What you get |
|---|---|
| `base.clone(name=..., instructions=...)` | Persona variant — model and all other settings inherited |
| `base.clone(model="gpt-5.5")` | Model upgrade — settings auto-update if they were at defaults |
| `base.clone(model_settings=ModelSettings(...))` | Explicit settings override — replaces inherited value entirely |
| `base.clone(tools=[*base.tools, new_tool])` | Extend tools safely — new list, shared tool references |

## Cell 5 — Basic Clone: Persona Variants

We define a single `base_agent` with a shared model, model settings, and a neutral writing instruction. Then we derive two persona variants — `pirate_agent` and `robot_agent` — by cloning the base and overriding only `name` and `instructions`.

All three agents use **exactly the same `model` and `model_settings`** from the base. We never touch those fields on the clones — they are inherited automatically.

We then run all three agents with the same prompt and compare their outputs side-by-side.

In [6]:
base_agent = Agent(
    name="Base Writer",
    instructions="You are a helpful writing assistant.",
    model=MODEL_NAME,
    model_settings=ModelSettings(
        reasoning=Reasoning(effort="none"),
        verbosity="low",
    ),
)

pirate_agent = base_agent.clone(
    name="Pirate Writer",
    instructions=(
        "You are a helpful writing assistant who always "
        "responds like a pirate."
    ),
)

robot_agent = base_agent.clone(
    name="Robot Writer",
    instructions=(
        "You are a helpful writing assistant who always "
        "responds like a robot. Use technical, precise language."
    ),
)

prompt = "Write a short description of a sunset."

result_base   = await Runner.run(base_agent,   prompt)
result_pirate = await Runner.run(pirate_agent, prompt)
result_robot  = await Runner.run(robot_agent,  prompt)

print("Base:\n",   result_base.final_output)
print("\nPirate:\n", result_pirate.final_output)
print("\nRobot:\n",  result_robot.final_output)

Base:
 The sunset spilled gold and crimson across the sky, fading slowly into soft violet as the day came to rest.

Pirate:
 Ahoy, the sun dips low o’er the sea, spillin’ gold and crimson across the sky. The clouds glow like embers, and the world grows quiet beneath the last warm light of day.

Robot:
 Sunset sequence initiated: the sky transitioned through amber, orange, and violet gradients as solar intensity diminished below the horizon. Clouds reflected residual light, producing a brief, high-saturation luminance event. End state: calm atmospheric fade.


## Cell 6 — Verify Independence: Originals Are Unchanged

A key guarantee of `clone()` is that it does **not** modify the original agent. Each clone is a fully independent object. Fields that are not overridden (like `model` and `model_settings`) point to the **same value** — not a copy of it — which is the expected shallow copy behaviour.

This cell inspects the `name`, `instructions`, and `model` of all three agents to confirm:
- Each agent has its own `name` and `instructions`.
- All three share the same `model` value (by inheritance, not by coincidence).

In [7]:
print("Base name:",   base_agent.name)
print("Pirate name:", pirate_agent.name)
print("Robot name:",  robot_agent.name)

print("\nBase instructions:",   base_agent.instructions[:40])
print("Pirate instructions:",   pirate_agent.instructions[:40])

print("\nAll use same model:", (
    base_agent.model == pirate_agent.model == robot_agent.model
))

Base name: Base Writer
Pirate name: Pirate Writer
Robot name: Robot Writer

Base instructions: You are a helpful writing assistant.
Pirate instructions: You are a helpful writing assistant who 

All use same model: True


## Cell 7 — Clone with Model Change: Auto `model_settings` Update

This cell demonstrates a behaviour that surprises many developers. When you clone an agent and pass a new `model` **without** passing `model_settings`, the SDK checks whether the current `model_settings` match the implicit defaults for the current model. If they do, it automatically resets `model_settings` to the implicit defaults for the **new** model.

**Why does this matter?** Different models in the GPT-5 family have different default reasoning efforts:

| Model | Default `reasoning.effort` |
|---|---|
| `gpt-5` | `"low"` |
| `gpt-5.5` | `"none"` |
| `gpt-5.4-mini` | `"none"` |

To make the auto-update **visible**, we use `gpt-5` as the base (default effort `"low"`) and clone to `gpt-5.5` (default effort `"none"`). The SDK detects that the base agent's `model_settings` are the implicit defaults for `gpt-5`, and automatically updates them to the implicit defaults for `gpt-5.5`. You can see the `reasoning.effort` change in the output.

**Key caveat:** If you pass both `model` and `model_settings` explicitly, the auto-update does **not** trigger — your explicit settings win. We will demonstrate that in Cell 8.

In [8]:
# gpt-5 defaults to reasoning.effort='low'
# gpt-5.5 defaults to reasoning.effort='none'
# This difference makes the auto-update visible in the output.
BASE_MODEL     = "gpt-5"
UPGRADED_MODEL = "gpt-5.5"

base_gpt5 = Agent(
    name="GPT-5 Agent",
    instructions="You are a helpful assistant.",
    model=BASE_MODEL,
)

upgraded_agent = base_gpt5.clone(
    name="GPT-5.5 Agent",
    model=UPGRADED_MODEL,
    # model_settings is NOT passed — auto-update will trigger
)

print("Base model:    ", base_gpt5.model)
print("Base effort:   ", base_gpt5.model_settings.reasoning.effort)
print()
print("Upgraded model:  ", upgraded_agent.model)
print("Upgraded effort: ", upgraded_agent.model_settings.reasoning.effort)
print()
print("# SDK auto-updated reasoning.effort from 'low' (gpt-5 default)")
print("# to 'none' (gpt-5.5 default) — because model_settings was not passed.")

Base model:     gpt-5
Base effort:    low

Upgraded model:   gpt-5.5
Upgraded effort:  none

# SDK auto-updated reasoning.effort from 'low' (gpt-5 default)
# to 'none' (gpt-5.5 default) — because model_settings was not passed.


## Cell 8 — Explicit `model_settings` Override: The Gotcha and the Fix

When you pass `model_settings` explicitly in a `clone()` call, the SDK's auto-update logic is bypassed entirely — this is hardcoded in `clone()`'s source:

```python
# from src/agents/agent.py
if "model" in kwargs and "model_settings" not in kwargs and ...:
    kwargs["model_settings"] = _initial_model_settings_for_model(kwargs["model"])
return dataclasses.replace(self, **kwargs)
```

The moment `model_settings` appears in your kwargs, that entire branch is skipped. `dataclasses.replace` then sets the clone's `model_settings` to **exactly the object you passed** — nothing more.

This means:
- The **base agent's** `model_settings` (e.g. `effort="low"` from `gpt-5`) is not consulted.
- The **new model's** implicit defaults (e.g. `effort="none"` for `gpt-5.5`) are not applied.
- Every field you did not specify in your `ModelSettings(...)` is `None` on the clone.

**`None` means the field is omitted from the API request entirely** — the server applies its own default. So `effort=None` is not the same as `effort="none"`. You have given up explicit control of that field.

This cell has two parts:
- **Part A** — the gotcha: base has `gpt-5` (`effort="low"`), clone to `gpt-5.5` with partial `model_settings`. What does `effort` end up as?
- **Part B** — the fix: specify every field you want. Treat `ModelSettings` as a complete specification, not a diff.

In [9]:
# --- Part A: The Gotcha ---
# base_gpt5 has model='gpt-5' → model_settings has effort='low'
# We clone to gpt-5.5 and pass model_settings — but only set temperature.
# The auto-update is skipped. The base agent's settings are not consulted.
# The clone gets exactly ModelSettings(temperature=0.5, verbosity='low') — nothing else.

gotcha_agent = base_gpt5.clone(
    name="Gotcha Agent",
    model=UPGRADED_MODEL,           # gpt-5.5
    model_settings=ModelSettings(
        temperature=0.5,
        verbosity="low",
        # reasoning.effort NOT specified
    ),
)

print("Base model:          ", base_gpt5.model)
print("Base effort:         ", base_gpt5.model_settings.reasoning.effort)
print()
print("Gotcha agent model:  ", gotcha_agent.model)
print("Gotcha agent effort: ", gotcha_agent.model_settings.reasoning)
print()
print("# effort is None — not 'low' from the base, not 'none' from gpt-5.5 defaults.")
print("# The auto-update was skipped. The clone got exactly what we passed — nothing else.")

Base model:           gpt-5
Base effort:          low

Gotcha agent model:   gpt-5.5
Gotcha agent effort:  None

# effort is None — not 'low' from the base, not 'none' from gpt-5.5 defaults.
# The auto-update was skipped. The clone got exactly what we passed — nothing else.


In [10]:
# --- Part B: The Fix ---
# Treat ModelSettings as a complete specification, not a diff.
# Specify every field you want — the clone gets exactly this object.

precise_agent = base_gpt5.clone(
    name="Precise Agent",
    model=UPGRADED_MODEL,           # gpt-5.5
    model_settings=ModelSettings(
        reasoning=Reasoning(effort="none"),  # explicitly set for gpt-5.5
        verbosity="low",
        temperature=0.5,
    ),
)

print("Precise agent model:       ", precise_agent.model)
print("Precise agent effort:      ", precise_agent.model_settings.reasoning.effort)
print("Precise agent temperature: ", precise_agent.model_settings.temperature)
print()
print("# All three fields are exactly what we specified.")
print("# When you pass model_settings, you own the entire object — specify everything you want.")

Precise agent model:        gpt-5.5
Precise agent effort:       none
Precise agent temperature:  0.5

# All three fields are exactly what we specified.
# When you pass model_settings, you own the entire object — specify everything you want.


## Cell 9 — Shallow Copy Warning: Mutable Tools List

When you pass `tools=[...]` to `clone()`, you get a **new list object**. That is good — modifying one agent's tool list will not affect the other. The tool function objects inside the list are shared references, but that is completely fine: tool functions are stateless, so sharing them across agents causes no problems.

The real danger is the opposite case: **not passing `tools` at all** when cloning. In that case, the clone and the original share the exact same list object. If you then call `agent.tools.append(new_tool)` on either agent, you accidentally mutate both.

This cell demonstrates both cases:
- **Part A** — pass `tools=[...]` → new list, safe to modify independently.
- **Part B** — don't pass `tools` → shared list, mutation affects both agents.

**The safe extend pattern** when you want to add a tool on a clone:
```python
new_agent = old_agent.clone(tools=[*old_agent.tools, new_tool])
```
Always pass a new list. Never mutate `agent.tools` directly.

> **Note on `function_tool`:** Used here only for demonstration. Full teaching is in Section 3.

In [11]:
@function_tool
def get_time() -> str:
    """Returns the current time."""
    from datetime import datetime
    return datetime.now().strftime("%H:%M:%S")

@function_tool
def get_date() -> str:
    """Returns today's date."""
    from datetime import datetime
    return datetime.now().strftime("%Y-%m-%d")

# --- Part A: Pass tools=[...] → new list object, safe ---
agent_a = base_agent.clone(
    name="Agent A",
    tools=[get_time],
)
print("Part A — tools passed explicitly:")
print("  Same list as base_agent.tools?  ", agent_a.tools is base_agent.tools)
print("  Same tool object (get_time)?    ", agent_a.tools[0] is get_time)
print("  ^ Shared ref — fine, tools are stateless functions.")

# Extend safely: unpack existing tools, add a distinct new tool
agent_b = agent_a.clone(
    name="Agent B",
    tools=[*agent_a.tools, get_date],
)
print("  agent_a tools:", [t.name for t in agent_a.tools])
print("  agent_b tools:", [t.name for t in agent_b.tools])
print("  get_time same object in both?", agent_a.tools[0] is agent_b.tools[0])
print("  ^ Still the same ref — expected and harmless.")

print()

# --- Part B: Don't pass tools → shared list object, dangerous ---
agent_c = base_agent.clone(
    name="Agent C",
    # tools NOT passed — clone shares base_agent.tools list object
)
print("Part B — tools NOT passed:")
print("  Same list as base_agent.tools?", agent_c.tools is base_agent.tools)
print("  Mutating agent_c.tools...")
agent_c.tools.append(get_time)
print("  agent_c tools:", [t.name for t in agent_c.tools])
print("  base_agent tools (also mutated!):", [t.name for t in base_agent.tools])

Part A — tools passed explicitly:
  Same list as base_agent.tools?   False
  Same tool object (get_time)?     True
  ^ Shared ref — fine, tools are stateless functions.
  agent_a tools: ['get_time']
  agent_b tools: ['get_time', 'get_date']
  get_time same object in both? True
  ^ Still the same ref — expected and harmless.

Part B — tools NOT passed:
  Same list as base_agent.tools? True
  Mutating agent_c.tools...
  agent_c tools: ['get_time']
  base_agent tools (also mutated!): ['get_time']


## Practical Cloning Patterns — Reference

Here is a quick-reference table of the most common cloning patterns you will use in real projects. None of these require writing a new `Agent(...)` from scratch — they all derive from a shared base.

| Pattern | How to do it |
|---|---|
| Persona variants | `base.clone(name=..., instructions=...)` |
| Model upgrade | `base.clone(model="gpt-5.5")` — settings auto-update if at defaults |
| Temperature tweak | `base.clone(model_settings=ModelSettings(temperature=0.0, ...))` |
| Add a tool to a clone | `base.clone(tools=[*base.tools, new_tool])` |
| Change output type | `base.clone(output_type=MyModel)` |

### What `clone()` does NOT cover in this notebook

- `as_tool()` — converting an agent into a tool (Section 3)
- `handoffs` — routing between agents (Section 5)
- `RealtimeAgent.clone()` — covered in Update Section U5
- Deep `output_type` cloning — only referenced in the table above